# Chapitre 3 - Exercices 3.26 à 3.30

| Exercice | Thème |
|---|---|
| 3.26 | Sélection ascendante et LAR : deux critères d'entrée différents |
| 3.27 | Conditions de Karush–Kuhn–Tucker du lasso, chemin linéaire par morceaux |
| 3.28 | Lasso avec une variable dupliquée |
| 3.29 | Ridge avec une variable dupliquée |
| 3.30 | L'elastic net comme un lasso sur des données augmentées |

Fonctions utilisées : `esl/ch03_regression_complements.py`.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl import ch03_regression_complements as rc

set_style()
np.set_printoptions(precision=4, suppress=True)

Les données du cancer de la prostate (échantillon d'apprentissage, 67 patients, 8
prédicteurs centrés-réduits) serviront d'exemple réel dans les exercices 3.27 à 3.29.

In [ ]:
pro = datasets.load_prostate()
pro_tr = pro[pro["train"]]
pnames = list(pro.columns[:8])
Xp, _, _, _ = rc.standardize(pro_tr[pnames].to_numpy(float))
yp = pro_tr["lpsa"].to_numpy(float)
yp = yp - yp.mean()
print(Xp.shape)

## Exercice 3.26 : Sélection ascendante contre LAR

**Énoncé (reformulé).** La sélection pas à pas ascendante (*forward stepwise*) fait entrer
à chaque étape la variable qui diminue le plus la somme des carrés résiduelle (RSS). Le LAR
fait progresser la variable de plus forte corrélation absolue avec le résidu courant.
Montrer que ces deux critères d'entrée ne sont pas nécessairement les mêmes.
*Indication :* notons $\mathbf x_{j\cdot\mathcal A}$ la variable $\mathbf x_j$ ajustée
linéairement sur les variables déjà dans le modèle ; montrer que le premier critère revient à
chercher le $j$ maximisant $|\operatorname{Corr}(\mathbf x_{j\cdot\mathcal A},\mathbf r)|$.

### Notations

Variables et réponse sont centrées ; les $\mathbf x_j$ sont de norme 1. Le modèle courant
contient les variables $\mathcal A$ ; $\mathbf H_{\mathcal A}$ est la projection orthogonale sur
$\operatorname{Vect}(\mathbf x_k,k\in\mathcal A)$ et $\mathbf r=\mathbf y-\mathbf H_{\mathcal A}\mathbf y$
le résidu (orthogonal aux variables actives). La variable ajustée est
$\mathbf x_{j\cdot\mathcal A}=\mathbf x_j-\mathbf H_{\mathcal A}\mathbf x_j$, orthogonale à
$\operatorname{Vect}(\mathbf x_k, k\in\mathcal A)$.

### Gain en RSS de la sélection ascendante

Ajouter $\mathbf x_j$ revient à projeter $\mathbf y$ sur
$\operatorname{Vect}(\mathbf x_k,k\in\mathcal A)\oplus\operatorname{Vect}(\mathbf x_{j\cdot\mathcal A})$
(même espace que avec $\mathbf x_j$, et somme **orthogonale**). La projection sur une somme
orthogonale est la somme des projections :

$$
\hat{\mathbf y}_{\mathcal A\cup\{j\}}=\mathbf H_{\mathcal A}\mathbf y+\frac{\langle\mathbf x_{j\cdot\mathcal A},\mathbf y\rangle}{\|\mathbf x_{j\cdot\mathcal A}\|^2}\mathbf x_{j\cdot\mathcal A}.
$$

Par Pythagore, la RSS diminue de

$$
\Delta_j=\frac{\langle\mathbf x_{j\cdot\mathcal A},\mathbf y\rangle^2}{\|\mathbf x_{j\cdot\mathcal A}\|^2}
=\frac{\langle\mathbf x_{j\cdot\mathcal A},\mathbf r\rangle^2}{\|\mathbf x_{j\cdot\mathcal A}\|^2}
=\|\mathbf r\|^2\cos^2(\mathbf x_{j\cdot\mathcal A},\mathbf r)
=\|\mathbf r\|^2\operatorname{Corr}^2(\mathbf x_{j\cdot\mathcal A},\mathbf r),
$$

où l'on a utilisé $\langle\mathbf x_{j\cdot\mathcal A},\mathbf H_{\mathcal A}\mathbf y\rangle=0$, et
le fait que pour des vecteurs centrés la corrélation est le cosinus de l'angle. Maximiser
$\Delta_j$ revient bien à maximiser $|\operatorname{Corr}(\mathbf x_{j\cdot\mathcal A},\mathbf r)|$.

### Critère du LAR

Le LAR choisit le $j$ maximisant $|\langle\mathbf x_j,\mathbf r\rangle|$. Comme $\mathbf r$ est
orthogonal aux variables actives, $\langle\mathbf x_j,\mathbf r\rangle=\langle\mathbf x_{j\cdot\mathcal A},\mathbf r\rangle$. En résumé, à résidu donné :

$$
\text{ascendante : }\max_j\frac{|\langle\mathbf x_{j\cdot\mathcal A},\mathbf r\rangle|}{\|\mathbf x_{j\cdot\mathcal A}\|},
\qquad
\text{LAR : }\max_j|\langle\mathbf x_{j\cdot\mathcal A},\mathbf r\rangle| .
$$

Les deux critères diffèrent par le facteur $1/\|\mathbf x_{j\cdot\mathcal A}\|\ge1$, d'autant plus
grand que $\mathbf x_j$ est **corrélée aux variables déjà présentes**. La sélection ascendante
favorise donc ces variables (une fois « nettoyées », elles peuvent expliquer beaucoup de
résidu), alors que le LAR les ignore. Les deux critères coïncident lorsque toutes les
$\|\mathbf x_{j\cdot\mathcal A}\|$ sont égales (par exemple si les prédicteurs sont orthogonaux,
ou à la première étape où $\mathcal A=\varnothing$).